In [1]:
# Step 1.필요한 라이브러리 로딩
import urllib.request , urllib
import pandas as pd    
from bs4 import BeautifulSoup
import os, tempfile, time, math
import undetected_chromedriver as uc   # pip install undetected-chromedriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import WebDriverException


In [2]:
#Step 2. 사용자에게 검색어 키워드를 입력 받습니다.
print("=" *80)
print(" 쿠팡 사이트의 식품 카테고리 Best Seller 상품 정보 추출하기 ")
print("=" *80)

cnt = int(input('1.크롤링 할 건수는 몇건입니까?: '))
page_cnt = math.ceil(cnt/60)

f_dir = input("2.파일을 저장할 폴더명만 쓰세요(기본경로:c:\\py_temp\\):")
if f_dir == '' :
    f_dir = "c:\\py_temp\\"
    
print("\n")

if cnt > 60 :
      print("요청 건수가 많아서 시간이 제법 소요되오니 잠시만 기다려 주세요~~")
else :
      print("요청하신 데이터를 수집하고 있으니 잠시만 기다려 주세요~~")



 쿠팡 사이트의 식품 카테고리 Best Seller 상품 정보 추출하기 


요청하신 데이터를 수집하고 있으니 잠시만 기다려 주세요~~


In [3]:
#Step 3.저장될 파일 경로와 이름을 지정합니다 # 이미지 파일, 텍스트 파일 따로 저장하고 나중에 합침
sec_name = '식품'
query_txt='쿠팡'

n = time.localtime()
s1 = '%04d-%02d-%02d-%02d-%02d-%02d' % (n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

os.makedirs(f_dir+s1+'-'+query_txt+'-'+sec_name)
os.chdir(f_dir+s1+'-'+query_txt+'-'+sec_name)

ff_dir=f_dir+s1+'-'+query_txt+'-'+sec_name
ff_name=f_dir+s1+'-'+query_txt+'-'+sec_name+'\\'+s1+'-'+query_txt+'-'+sec_name+'.txt'
fc_name=f_dir+s1+'-'+query_txt+'-'+sec_name+'\\'+s1+'-'+query_txt+'-'+sec_name+'.csv'
fx_name=f_dir+s1+'-'+query_txt+'-'+sec_name+'\\'+s1+'-'+query_txt+'-'+sec_name+'.xlsx'

# 제품 이미지 저장용 폴더 생성
img_dir = ff_dir+"\\images"
os.makedirs(img_dir)
os.chdir(img_dir)
    
s_time = time.time( )



In [4]:
# Step 4. 중요!!  undetected chromedriver 설정하기
CHROME_PATH = r"C:\Program Files\Google\Chrome\Application\chrome.exe"  # 크롬 경로 확인!
VERSION_MAIN = 145  # 현재 크롬 메이저 버전 ~ 버전이 바뀌면 그때 또 수정하면 됨 

options = Options()
# 필요하면 headless: 많은 사이트가 탐지하니 처음엔 꺼두고 테스트
# options.add_argument("--headless=new")

# 깨끗한 임시 프로필(기존 프로필 충돌 방지)
user_data = os.path.join(tempfile.gettempdir(), "uc_profile_clean")
os.makedirs(user_data, exist_ok=True)

driver = uc.Chrome(
    options=options,
    browser_executable_path=CHROME_PATH,
    version_main=VERSION_MAIN,      # ← 핵심: 140으로 맞추기
    user_data_dir=user_data,
    use_subprocess=False,           # 안 되면 True로 바꿔 재시도
    patcher_force_close=True,       # 패쳐가 점유 중일 때 강제 종료
    debug=True,
)
# 여기까지 그대로 복붙해서 쓰면 됨 
# 이전까지는 카테고리 누르고, 식품 눌렀는데, 이제는 링크로 이동하는구나?? 
driver.get("https://www.coupang.com/np/categories/194276")
driver.maximize_window()
time.sleep(3)



In [5]:
# driver.find_element(By.ID,'wa-category').click()
# # <a href="/np/categories/194276" data-text="식품" data-el="food" style="background-image: url(&quot;https://assets.coupangcdn.com/front/front-web-next/_next/static/media/icons.03b3790b.png&quot;);">식품<i class="si"></i></a>
# driver.find_element(By.XPATH, '//a[text()="식품"]').click()
# #driver.find_element(By.CSS_SELECTOR, 'a[href="/np/categories/194276"]').click()


In [6]:
#Step 5. 내용을 수집합니다
print("\n")
print("===== 곧 수집된 결과를 출력합니다 ^^ ===== ")
print("\n")

ranking2=[]        #제품의 판매순위 저장
title2=[]          #제품 정보 저장
p_price2=[]        #현재 판매가 저장
discount2 = []     #할인율 저장
sat_count2=[]      #상품평 수 저장
star2=[ ]          #상품별점

img_src2=[]   # 이미지 URL 저장변수
file_no = 0   # 이미지 파일 저장할 때 번호
count = 1     # 총 게시물 건수 카운트 변수

#쇼핑몰 / 아마존 / 쿠팡
#보통 scrollTo
# 이유 맨 아래 내려야 다음 상품 로딩

def scroll_down(driver):      
    driver.execute_script("window.scrollBy(0,1100);")
    time.sleep(1)

for x in range(1,page_cnt + 1) :
    
    for cc in range(1,7) :
        scroll_down(driver)
        
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    item_result = soup.find('div','ProductList_productList__kfbKM').find_all('li')

    for li in item_result :
        if cnt < count :
            break

        # 제품 이미지 다운로드 하기
        try :
            photo = li.find('figure','ProductUnit_productImage__Mqcg1').find('img')['src']
        except AttributeError :
            continue

        file_no += 1
        
        urllib.request.urlretrieve(photo,str(file_no)+'.jpg')
        time.sleep(0.5)

        #제품 내용 추출하기
        f = open(ff_name, 'a',encoding='UTF-8')
        f.write("-----------------------------------------------------"+"\n")
        print("-" *70)

        ranking = count
        print("1.판매순위:",ranking)
        f.write('1.판매순위:'+ str(ranking) + "\n")

        try :
            t = li.find('div','ProductUnit_productNameV2__cV9cw').get_text().replace("\n","")
        except :
            title = '제품이름 없습니다'
            print(title.replace("\n",""))
            f.write('2.제품이름:'+ title + "\n")
        else :
            title = t.replace("\n","").strip()
            print("2.제품이름:", title.replace("\n","").strip())                  
            f.write('2.제품이름:'+ title + "\n")

        try :
            p_price = li.find('strong','Price_priceValue__A4KOr').get_text().replace("\n","")
        except :
            p_price = '0'
            print("3.판매가격:", p_price.replace("\n",""))
            f.write('3.판매가격:'+ p_price + "\n")
        else :
            print("3.판매가격:", p_price.replace("\n",""))
            f.write('3.판매가격:'+ p_price + "\n")

        try :
            discount = li.find('span','PriceInfo_discountRate__EsQ8I').get_text().replace("\n","")
        except  :
            discount = '0%'
            print("4:할인률:", discount)
            f.write('4.할인율:'+ discount + "\n")
        else :
            print("4:할인률:", discount)
            f.write('4.할인율:'+ discount + "\n")

        try :
            sat_count_1 = li.find('span','ProductRating_ratingCount__R0Vhz').get_text()
            sat_count_2 = sat_count_1.replace("(","").replace(")","")
        except  :
            sat_count_2='0'
            print('5.상품평 수: ',sat_count_2)
            f.write('5.상품평 수:'+ sat_count_2 + "\n")
        else :
            print('5.상품평 수:',sat_count_2)
            f.write('5.상품평 수:'+ sat_count_2 + "\n")

        try : 
            star = li.find('span','ProductRating_rating__lMxS9').get_text()
        except :
            star = 0
            print('6.별점:',star)
            f.write('6.별점:'+ star + "\n") #            star2.append(star)
        else:
            print('6.별점:',star)
            f.write('6.별점:'+ star + "\n") #            star2.append(star)           

        print("-" *70)

        f.close( )             
        time.sleep(0.5)

        ranking2.append(ranking)
        title2.append(title.replace("\n",""))
        p_price2.append(p_price.replace("\n",""))
        discount2.append(discount)
        star2.append(star)

        try :   
            sat_count2.append(sat_count_2)
        except IndexError :
            sat_count2.append(0)

        count += 1
        
        time.sleep(1)
        
    x += 1       
    try :
        # # # Access Denied 메시지가 나오면 아래코드로 쿠키를 삭제한다
        # driver.delete_all_cookies()
        time.sleep(2)
        driver.find_element(By.LINK_TEXT, '%s' %x).click() # 다음 페이지번호 클릭
    except :
        break





===== 곧 수집된 결과를 출력합니다 ^^ ===== 


----------------------------------------------------------------------
1.판매순위: 1
2.제품이름: 농심 신라면 120g, 10개
3.판매가격: 8,300원
4:할인률: 0%
5.상품평 수: 776337
6.별점: 5
----------------------------------------------------------------------
----------------------------------------------------------------------
1.판매순위: 2
2.제품이름: 해태제과 에이스 과자, 364g, 2개
3.판매가격: 5,700원
4:할인률: 0%
5.상품평 수: 68288
6.별점: 4.5
----------------------------------------------------------------------
----------------------------------------------------------------------
1.판매순위: 3
2.제품이름: 불닭볶음면 까르보 큰컵 105g, 6개
3.판매가격: 8,100원
4:할인률: 10%
5.상품평 수: 45652
6.별점: 5
----------------------------------------------------------------------
----------------------------------------------------------------------
1.판매순위: 4
2.제품이름: 육개장 사발면 86g, 12개
3.판매가격: 9,500원
4:할인률: 8%
5.상품평 수: 473340
6.별점: 5
----------------------------------------------------------------------
-------------------------------------------------

In [7]:
#step 6. csv , xlsx 형태로 저장하기              
co_best_seller = pd.DataFrame()
co_best_seller['판매순위']=ranking2
co_best_seller['제품소개']=pd.Series(title2)
co_best_seller['제품판매가']=pd.Series(p_price2)
co_best_seller['할인율']=pd.Series(discount2)
co_best_seller['상품평수']=pd.Series(sat_count2)
co_best_seller['별점']=pd.Series(star2)

# csv 형태로 저장하기
co_best_seller.to_csv(fc_name,encoding="utf-8-sig",index=False)

# 엑셀 형태로 저장하기
co_best_seller.to_excel(fx_name ,index=False , engine='openpyxl')

e_time = time.time( )
t_time = e_time - s_time

count -= 1
print("\n")
print("=" *100)
print("1.요청된 총 %s 건의 리뷰 중에서 실제 크롤링 된 리뷰수는 %s 건입니다" %(cnt,count))
print("2.총 소요시간은 %s 초 입니다 " %round(t_time,1))
print("3.파일 저장 완료: txt 파일명 : %s " %ff_name)
print("4.파일 저장 완료: csv 파일명 : %s " %fc_name)
print("5.파일 저장 완료: xlsx 파일명 : %s " %fx_name)
print("=" *100)





1.요청된 총 10 건의 리뷰 중에서 실제 크롤링 된 리뷰수는 10 건입니다
2.총 소요시간은 38.5 초 입니다 
3.파일 저장 완료: txt 파일명 : c:\py_temp\2026-03-10-18-15-59-쿠팡-식품\2026-03-10-18-15-59-쿠팡-식품.txt 
4.파일 저장 완료: csv 파일명 : c:\py_temp\2026-03-10-18-15-59-쿠팡-식품\2026-03-10-18-15-59-쿠팡-식품.csv 
5.파일 저장 완료: xlsx 파일명 : c:\py_temp\2026-03-10-18-15-59-쿠팡-식품\2026-03-10-18-15-59-쿠팡-식품.xlsx 


In [8]:
#Step 7. xlsx 파일에 제품 이미지 삽입하기 - Open office 의 경우 사용 코드
# pip install openpyxl pillow
import PIL
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage

# 기존에 저장한 xlsx 열기
wb = load_workbook(fx_name)
ws = wb["Sheet1"]  # pandas의 기본 시트명

# 열/행 크기 조정 (열 너비는 문자 폭 기준, 행 높이는 포인트 기준)
ws.column_dimensions['B'].width = 25   # 이미지 가로에 맞춰 넉넉히 # b 컬럼 넓이 
row_cnt = cnt + 1
for r in range(2, row_cnt + 1):        # 2행 ~ cnt+1행 , 1행은 제목이니까 2행부터 높아지면 됨
    ws.row_dimensions[r].height = 100  # 대략 120pt ≈ 160px 내외 # 컬럼 높이 

# C2 ~ C(cnt+1)에 이미지 삽입 (1.jpg ~ cnt.jpg)
for i in range(1, cnt + 1):
    cell_addr = f"B{i+1}"
    img_path = os.path.join(img_dir, f"{i}.jpg")
    if not os.path.exists(img_path):
        continue

    img = XLImage(img_path)
    # 표시 크기(px). openpyxl은 픽셀 단위를 내부 pt로 변환해 저장합니다.
    img.width = 130
    img.height = 100

    # 셀 기준으로 고정 배치
    ws.add_image(img, cell_addr)

# 저장
wb.save(fx_name)


# #Step 7_2. xlsx 파일에 제품 이미지 삽입하기 - MS Excel 의 경우 사용 코드
# import win32com.client as win32   #pywin32 , pypiwin32 설치후 동작
# import win32api                   #파이썬 프롬프트를 관리자 권한으로 실행해야 에러없음
                     
# excel = win32.Dispatch('Excel.Application')
# wb = excel.Workbooks.Open(fx_name)
# sheet = wb.ActiveSheet
# sheet.Columns(2).ColumnWidth = 30   
# row_cnt = cnt+1
# sheet.Rows("2:%s" %row_cnt).RowHeight = 120   

# ws = wb.Sheets("Sheet1")
# col_name2=[]
# file_name2=[]

# for a in range(2,cnt+2) :
#     col_name='B'+str(a)
#     col_name2.append(col_name)

# for b in range(1,cnt+1) :
#     file_name=img_dir+'\\'+str(b)+'.jpg'
#     file_name2.append(file_name)
      
# for i in range(0,cnt) :
#     rng = ws.Range(col_name2[i])
#     image = ws.Shapes.AddPicture(file_name2[i], False, True, rng.Left, rng.Top, 130, 100)
#     excel.Visible=True
#     excel.ActiveWorkbook.Save()

# #driver.close( )

In [9]:
os.getcwd()

'c:\\py_temp\\2026-03-10-18-15-59-쿠팡-식품\\images'

In [10]:
# 쿠팡 네이버 올리브영 
